# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Dataset Croissant schema URL:**
```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```


In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print a basic description
print("Dataset title:", metadata.name)
print("Description:", metadata.description)
print("Identifier:", metadata.identifier)
print("Published:", metadata.datePublished)
print("RecordSet count:", len(metadata.recordSet))

## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset provides three distinct tables:
1. Table 1 – Clinical features
2. Table 2 – Hormone levels and imaging
3. Table 3 – Genetic variants

We will enumerate the available record sets using their `@id`, then inspect fields for each table.

In [ ]:
# List the available RecordSets and their IDs
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    # Try using the Croissant schema directly if not available in metadata
    import requests
    croissant_schema = requests.get(croissant_url).json()
    record_sets = croissant_schema.get('recordSet', [])

if isinstance(record_sets, dict):
    # Single recordset case
    record_sets = [record_sets]

record_set_ids = []

print("Available record sets:")
for rs in record_sets:
    rs_id = rs.get('@id') if isinstance(rs, dict) else rs
    record_set_ids.append(rs_id)
    print(f"- RecordSet @id: {rs_id}")

# For demonstration, print fields/columns for each record set (if present)
for rs in record_sets:
    rs_id = rs.get('@id') if isinstance(rs, dict) else rs
    print(f"\nRecordSet {rs_id}: Fields/Columns:")
    fields = rs.get('field', []) if isinstance(rs, dict) else []
    if fields:
        if isinstance(fields, dict): fields = [fields]
        for field in fields:
            print(f"  - Field @id: {field.get('@id')} | name: {field.get('name')} | dataType: {field.get('dataType')}")
    else:
        print("  [Fields not available in metadata, will inspect after loading records]")

## 3. Data Extraction
Load data from each record set into DataFrames for analysis.

All entities are referenced using their `@id`.

We will use the discovered record set IDs and dynamically extract all records for each.

In [ ]:
# Create a mapping of record set ids to DataFrames
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for RecordSet {record_set_id}")
        print("Columns:", df.columns.tolist())
        print("Sample records:")
        print(df.head(2))
    except Exception as e:
        print(f"Could not load records for RecordSet {record_set_id}: {e}")

# Show summary of loaded tables
print("\nSummary of loaded DataFrames:")
for rid, df in dataframes.items():
    print(f"- {rid}: {len(df)} rows, {len(df.columns)} columns")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

We'll focus on Table 1 (Clinical features) and Table 2 (Hormone levels), referencing columns by their `@id`.

In [ ]:
# Select Table 1 (clinical features) and Table 2 (hormone levels)
# Adjust these IDs to match actual recordSet @id values as detected earlier
table1_id = record_set_ids[0] if len(record_set_ids) > 0 else None
table2_id = record_set_ids[1] if len(record_set_ids) > 1 else None

df1 = dataframes.get(table1_id, pd.DataFrame())  # Table 1
df2 = dataframes.get(table2_id, pd.DataFrame())  # Table 2

if not df1.empty:
    # Example: Filter patients older than 30 years at diagnosis
    # Find a column @id like 'age_at_diagnosis' (replace with correct @id)
    age_id = None
    numeric_cols = [col for col in df1.columns if 'age' in col or 'height' in col or df1[col].dtype in ['int64','float64']]
    if numeric_cols:
        age_id = numeric_cols[0]

    if age_id:
        threshold = 30
        filtered_df = df1[df1[age_id] > threshold]
        print(f"Filtered records from Table 1 with {age_id} > {threshold}:")
        print(filtered_df)

        # Normalize the numeric field
        filtered_df[f"{age_id}_normalized"] = (filtered_df[age_id] - filtered_df[age_id].mean()) / filtered_df[age_id].std()
        print(f"Normalized {age_id} for filtered records:")
        print(filtered_df[[age_id, f"{age_id}_normalized"]])

        # Group by categorical field (e.g. 'infertility' or 'hirsutism', using their column @id)
        group_fields = [col for col in df1.columns if ('infertility' in col or 'hirsutism' in col or df1[col].dtype == 'object')]
        group_field = group_fields[0] if group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[age_id].mean().reset_index()
            print(f"Grouped ages by {group_field} in Table 1:")
            print(grouped_df)
else:
    print("Table 1 dataframe is not available for EDA.")

if not df2.empty:
    # Example: Analyze hormone values above threshold (pick a numeric hormone field @id)
    hormone_cols = [col for col in df2.columns if df2[col].dtype in ['int64','float64']]
    hormone_id = hormone_cols[0] if hormone_cols else None
    if hormone_id:
        hormone_threshold = df2[hormone_id].mean()
        filtered_hormone = df2[df2[hormone_id] > hormone_threshold]
        print(f"Hormone records with {hormone_id} above mean ({hormone_threshold:.2f}):")
        print(filtered_hormone[[hormone_id]])
else:
    print("Table 2 dataframe is not available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot numeric distributions from Table 1 and Table 2.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Table 1: plot age or height distribution if available
if not df1.empty:
    numeric_cols = [col for col in df1.columns if df1[col].dtype in ['int64','float64']]
    if numeric_cols:
        field = numeric_cols[0]
        plt.figure(figsize=(6, 4))
        sns.histplot(df1[field], bins=10, kde=True)
        plt.title(f"Distribution of {field} (Table 1)")
        plt.xlabel(field)
        plt.ylabel("Count")
        plt.show()

# Table 2: plot first hormone distribution if available
if not df2.empty:
    hormone_cols = [col for col in df2.columns if df2[col].dtype in ['int64','float64']]
    if hormone_cols:
        field = hormone_cols[0]
        plt.figure(figsize=(6, 4))
        sns.histplot(df2[field], bins=10, kde=True)
        plt.title(f"Distribution of {field} (Table 2)")
        plt.xlabel(field)
        plt.ylabel("Count")
        plt.show()

## 6. Conclusion
We explored the FAIR^2 clinical dataset using `mlcroissant`, loading metadata and record sets by their `@id`, and performed basic data analysis and visualization. Key observations include:
- The dataset consists of three tables capturing clinical features, hormone levels, imaging, and genotype information for three female patients with NC-21OHD-associated infertility.
- Using column and record set `@id` references ensures reproducibility and clarity in downstream analysis.
- Dataset size is limited (N=3), appropriate for demonstrations and academic analyses, but not suitable for ML model training.

For further analysis, consult domain knowledge to interpret and re-use the dataset following ethical guidelines and research recommendations.